# 03 · Componente no supervisado — segmentación de clientes
**Objetivo:** descubrir perfiles de clientes **sin usar `y`** y analizar después su relación con la
aceptación del depósito.

## 1. Selección y justificación de algoritmos
| Algoritmo | Por qué | Supuestos | Ventajas | Limitaciones |
|---|---|---|---|---|
| K-Means | Estándar para segmentación | Clusters esféricos de tamaño similar | Rápido, interpretable | Sensible a escala y atípicos; hay que fijar k |
| Mezcla gaussiana (GMM) | Clusters elípticos y asignación probabilística | Datos ≈ mezcla de gaussianas | Más flexible que K-Means | Más costoso, sensible a inicialización |
| Jerárquico (Ward) | Verificar la estabilidad de la estructura | Clusters compactos | Dendrograma, sin inicialización | O(n²): requiere muestra |

_(Ajustar con los argumentos del equipo; ver `docs/03_modelos_no_supervisados.md`.)_

In [ ]:
import sys
sys.path.append("..")   # permite `from src import ...` al ejecutar desde notebooks/

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import config
from src.data_loader import get_train_test, load_data

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
config.FIGURES_DIR.mkdir(parents=True, exist_ok=True)
config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)

X_train, X_test, y_train, y_test = get_train_test()
print("Train:", X_train.shape, "| Test:", X_test.shape)

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import adjusted_rand_score
from src import unsupervised as uns

## 2. Matriz de trabajo
Preprocesamiento ajustado con entrenamiento + **PCA (90 % de varianza)** para que las distancias tengan
sentido con variables mezcladas (numéricas + one-hot). `y` no entra en este paso.

In [ ]:
Z, _, prep, pca = uns.prepare_matrix(X_train)
print("Dimensión tras PCA:", Z.shape, "| varianza explicada:", pca.explained_variance_ratio_.sum().round(3))

## 3. Elección de k (codo + Silhouette + Davies-Bouldin)

In [ ]:
sweep = uns.sweep_kmeans(Z, range(2, 11))
display(sweep.round(3))
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(sweep["k"], sweep["inercia"], marker="o"); axes[0].set_title("Codo (inercia)")
axes[1].plot(sweep["k"], sweep["silhouette"], marker="o", color="green"); axes[1].set_title("Silhouette (↑)")
axes[2].plot(sweep["k"], sweep["davies_bouldin"], marker="o", color="red"); axes[2].set_title("Davies-Bouldin (↓)")
for ax in axes: ax.set_xlabel("k")
plt.tight_layout(); plt.savefig(config.FIGURES_DIR / "uns_eleccion_k.png", dpi=150); plt.show()
sweep.to_csv(config.RESULTS_DIR / "no_supervisado_barrido_k.csv", index=False)

**Decisión (completar):** k elegido y justificación (no solo el máximo Silhouette: también interpretabilidad y
tamaño de los grupos): …

In [ ]:
K = 4   # <- AJUSTAR según el análisis anterior

## 4. Entrenamiento y comparación de algoritmos

In [ ]:
km = uns.fit_kmeans(Z, K);  labels_km = km.labels_
gmm = uns.fit_gmm(Z, K);    labels_gmm = gmm.predict(Z)
idx_h, labels_h = uns.fit_agglomerative(Z, K)       # sobre muestra

res = pd.DataFrame({
    "K-Means": uns.clustering_metrics(Z, labels_km),
    "GMM": uns.clustering_metrics(Z, labels_gmm),
    "Jerárquico (muestra)": uns.clustering_metrics(Z[idx_h], labels_h),
}).T
display(res.round(3))
res.to_csv(config.RESULTS_DIR / "no_supervisado_comparacion.csv")
print("Acuerdo K-Means vs GMM (ARI):", round(adjusted_rand_score(labels_km, labels_gmm), 3))
print("Acuerdo K-Means vs Jerárquico (ARI, en la muestra):", round(adjusted_rand_score(labels_km[idx_h], labels_h), 3))

**Interpretación (completar):** ¿qué algoritmo separa mejor? ¿Los grupos son estables entre algoritmos (ARI)?
Silhouette bajo suele indicar que no hay clusters muy separados, sino un continuo: …

## 5. Optimización y estabilidad de los modelos
- **GMM:** elegimos el tipo de covarianza y k con **BIC/AIC** (criterios de información: penalizan la complejidad).
- **Jerárquico:** comparamos criterios de enlace (ward, average, complete) sobre una muestra.
- **K-Means:** medimos la **estabilidad** repitiendo con distintas semillas (ARI = 1 → mismos grupos siempre).

In [ ]:
bic = uns.sweep_gmm_bic(Z, range(2, 9))
display(bic.pivot(index="k", columns="covarianza", values="bic").round(0))
for cov, g in bic.groupby("covarianza"):
    plt.plot(g["k"], g["bic"], marker="o", label=cov)
plt.xlabel("k"); plt.ylabel("BIC (menor = mejor)"); plt.legend(title="covarianza"); plt.title("GMM: BIC por k y tipo de covarianza")
plt.savefig(config.FIGURES_DIR / "uns_gmm_bic.png", dpi=150, bbox_inches="tight"); plt.show()
bic.to_csv(config.RESULTS_DIR / "no_supervisado_gmm_bic.csv", index=False)

In [ ]:
link = uns.compare_linkages(Z, K, sample_size=4000)
display(link.round(3))
stab = uns.kmeans_stability(Z, K, n_runs=10)
print("Estabilidad K-Means (ARI entre 10 semillas):", {k: round(v, 3) for k, v in stab.items()})
link.to_csv(config.RESULTS_DIR / "no_supervisado_linkages.csv")

**Interpretación (completar):** ¿qué covarianza/k prefiere el BIC y coincide con el k de K-Means? ¿Qué enlace
produce grupos más equilibrados (ojo con un cluster que absorbe casi todo)? ¿Son estables los grupos?: …

## 6. Visualización y reducción de dimensionalidad

In [ ]:
rng = np.random.RandomState(config.RANDOM_STATE)
s = rng.choice(len(Z), size=3000, replace=False)           # muestra para t-SNE
pca2 = PCA(n_components=2, random_state=config.RANDOM_STATE)
P = pca2.fit_transform(Z)
tsne = TSNE(n_components=2, perplexity=30, random_state=config.RANDOM_STATE, init="pca").fit_transform(Z[s])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sc0 = axes[0].scatter(P[s, 0], P[s, 1], c=labels_km[s], cmap="tab10", s=6); axes[0].set_title(f"PCA 2D · K-Means (k={K})")
sc1 = axes[1].scatter(tsne[:, 0], tsne[:, 1], c=labels_km[s], cmap="tab10", s=6); axes[1].set_title("t-SNE · K-Means")
sc2 = axes[2].scatter(tsne[:, 0], tsne[:, 1], c=y_train.values[s], cmap="coolwarm", s=6); axes[2].set_title("t-SNE · coloreado por y (solo interpretación)")
for ax, sc in [(axes[0], sc0), (axes[1], sc1)]:
    ax.legend(*sc.legend_elements(), title="cluster", fontsize=8)
axes[2].legend(sc2.legend_elements()[0], ["no", "yes"], title="y", fontsize=8)
plt.tight_layout(); plt.savefig(config.FIGURES_DIR / "uns_visualizacion.png", dpi=150); plt.show()
print("Varianza explicada por los 2 primeros componentes (sobre Z):", pca2.explained_variance_ratio_.sum().round(3))

**Interpretación (completar):** ¿qué información se conserva y cuál se pierde en 2D? (t-SNE no conserva
distancias globales; PCA 2D explica solo parte de la varianza): …

## 7. Perfiles de clientes

In [ ]:
profile = uns.profile_clusters(X_train, labels_km)
display(profile.T)
profile.to_csv(config.RESULTS_DIR / "no_supervisado_perfiles.csv")

In [ ]:
# Perfil visual: media estandarizada de las variables numéricas por cluster
num = X_train.select_dtypes(include=np.number)
zmean = ((num - num.mean()) / num.std()).assign(cluster=labels_km).groupby("cluster").mean()
plt.figure(figsize=(9, 4)); sns.heatmap(zmean, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Perfil de clusters (z-score por variable numérica)")
plt.savefig(config.FIGURES_DIR / "uns_perfiles.png", dpi=150, bbox_inches="tight"); plt.show()

**Perfiles (completar):** nombre y descripción de cada segmento (p. ej. "jóvenes sin crédito", "clientes
recontactados con balance alto"…) con base en la tabla y el mapa de calor: …

## 8. Relación entre los segmentos y la aceptación (ahora sí se usa `y`)

In [ ]:
tabla, chi = uns.cluster_vs_target(labels_km, y_train)
display(tabla.round(3))
print(f"Tasa global de aceptación: {y_train.mean():.3f} | chi2={chi['chi2']:.1f}, p={chi['p_valor']:.2e}")
tabla["tasa_aceptacion"].plot(kind="bar", color="#4C72B0", rot=0)
plt.axhline(y_train.mean(), color="red", ls="--"); plt.title("Tasa de aceptación por segmento")
plt.savefig(config.FIGURES_DIR / "uns_aceptacion_por_cluster.png", dpi=150, bbox_inches="tight"); plt.show()
tabla.to_csv(config.RESULTS_DIR / "no_supervisado_aceptacion.csv")

## 9. Conclusiones y recomendaciones de campaña (completar)
- Segmentos con mayor y menor aceptación y por qué: …
- Recomendaciones para focalizar campañas (a qué segmentos priorizar, canal, mes): …
- Limitaciones (Silhouette, variables mezcladas, estabilidad, k elegido): …
- ¿Descubrió el modelo estructuras no evidentes en el EDA?: …